In [ ]:
# lab3_unsupervised_wine.py
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

# 1. Load + normalize
data = load_wine()
X = data['data']
y = data['target']
feature_names = data['feature_names']

scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

# 2. KMeans sweep k=2..10
k_range = range(2, 11)
records = []
for k in k_range:
    km = KMeans(n_clusters=k, init='random', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertia = float(km.inertia_)
    sil = float(silhouette_score(X_scaled, labels)) if len(set(labels))>1 else np.nan
    records.append({'k':k, 'inertia':inertia, 'silhouette':sil})
kmeans_df = pd.DataFrame(records)
# delta inertia
wcss = kmeans_df['inertia'].values
delta = [0.0]
for i in range(1, len(wcss)):
    delta.append( (wcss[i-1] - wcss[i]) / wcss[i-1] )
kmeans_df['delta_inertia'] = delta
kmeans_df.to_csv('kmeans_metrics.csv', index=False)

# Plots: inertia, delta, silhouette
plt.figure(); plt.plot(kmeans_df['k'], kmeans_df['inertia'], marker='o'); plt.title('Inertia'); plt.xlabel('k'); plt.grid(True); plt.savefig('inertia_by_k.png')
plt.figure(); plt.plot(kmeans_df['k'], kmeans_df['delta_inertia'], marker='o'); plt.title('Delta inertia'); plt.xlabel('k'); plt.grid(True); plt.savefig('delta_by_k.png')
plt.figure(); plt.plot(kmeans_df['k'], kmeans_df['silhouette'], marker='o'); plt.title('Silhouette'); plt.xlabel('k'); plt.grid(True); plt.savefig('silhouette_by_k.png')

# 3. DBSCAN grid search (eps, min_samples) and visualization (PCA(2) for plotting)
eps_list = [0.2, 0.3, 0.5, 0.7, 1.0]
min_s_list = [3,5,7,10]
db_recs = []
for eps in eps_list:
    for ms in min_s_list:
        db = DBSCAN(eps=eps, min_samples=ms)
        labels = db.fit_predict(X_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        sil = silhouette_score(X_scaled, labels) if n_clusters > 1 else np.nan
        noise = int(np.sum(labels == -1))
        db_recs.append({'eps':eps, 'min_samples':ms, 'n_clusters':n_clusters, 'silhouette':sil, 'noise_count':noise})
dbscan_df = pd.DataFrame(db_recs)
dbscan_df.to_csv('dbscan_grid.csv', index=False)

# pick the best DBSCAN run by silhouette (if any have >1 cluster)
valid = dbscan_df.dropna(subset=['silhouette'])
if not valid.empty:
    best = valid.loc[valid['silhouette'].idxmax()]
else:
    best = dbscan_df.iloc[0]

# plot DBSCAN result in PCA(2)
pca2 = PCA(n_components=2).fit_transform(X_scaled)
db = DBSCAN(eps=float(best['eps']), min_samples=int(best['min_samples']))
labels_db = db.fit_predict(X_scaled)
plt.figure(figsize=(6,5))
for lbl in np.unique(labels_db):
    mask = (labels_db == lbl)
    px = pca2[mask, 0]; py = pca2[mask,1]
    marker = 'x' if lbl == -1 else 'o'
    plt.scatter(px, py, marker=marker, label=('noise' if lbl==-1 else f'cluster {lbl}'))
plt.legend(); plt.title(f"DBSCAN eps={best['eps']} min_samples={best['min_samples']}"); plt.grid(True)
plt.savefig('dbscan_pca2.png')

# list outlier indices
outliers = np.where(labels_db == -1)[0].tolist()

# 4. PCA explained variance
pca = PCA().fit(X_scaled)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_comp_90 = int(np.searchsorted(cumvar, 0.90) + 1)

plt.figure(); plt.plot(range(1, len(cumvar)+1), cumvar, marker='o'); plt.title('PCA cumulative explained variance'); plt.grid(True); plt.savefig('pca_cumvar.png')
# save component loadings table
df_components = pd.DataFrame(pca.components_, columns=feature_names, index=[f'PC{i+1}' for i in range(pca.components_.shape[0])])
df_components.to_csv('pca_components.csv')

# 5. Compare KMeans before vs after PCA (use best k by silhouette from earlier)
best_k = int(kmeans_df.loc[kmeans_df['silhouette'].idxmax()]['k'])
km_orig = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42).fit(X_scaled)
labels_orig = km_orig.labels_
km_pca = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42).fit(PCA(n_components=n_comp_90).fit_transform(X_scaled))
labels_pca = km_pca.labels_

metrics_cmp = {
    'orig_inertia': km_orig.inertia_,
    'orig_silhouette': silhouette_score(X_scaled, labels_orig),
    'pca_inertia': km_pca.inertia_,
    'pca_silhouette': silhouette_score(PCA(n_components=n_comp_90).fit_transform(X_scaled), labels_pca),
    'n_components_90': n_comp_90
}
pd.Series(metrics_cmp).to_csv('kmeans_pca_comparison.csv')

# 6. KMeans variation: MiniBatchKMeans comparison (time + silhouette)
from sklearn.cluster import MiniBatchKMeans
results = []
for name, model in [('KMeans', KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42)),
                    ('MiniBatchKMeans', MiniBatchKMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42, batch_size=32))]:
    start = time.time()
    labels = model.fit_predict(X_scaled)
    elapsed = time.time() - start
    results.append({'method':name, 'silhouette':silhouette_score(X_scaled, labels), 'time':elapsed, 'inertia':float(model.inertia_)})
pd.DataFrame(results).to_csv('kmeans_variant_comparison.csv')

# Print summary for quick view:
print("KMeans (k sweep) table saved to kmeans_metrics.csv")
print("DBSCAN grid saved to dbscan_grid.csv; chosen example:", best.to_dict())
print("DBSCAN outliers count:", len(outliers), "indices (first 10):", outliers[:10])
print("PCA n_components >=90% variance:", n_comp_90)
print("KMeans vs PCA comparison saved to kmeans_pca_comparison.csv")
print("KMeans variant comparison saved to kmeans_variant_comparison.csv")